# Secondary specificity validation on non-CVA Voisard cohorts

This is a model-validation experiment, not a new data-source investigation. The primary CNN was trained only for healthy-versus-CVA classification. Here, the frozen fold checkpoints are applied to the six other complete Voisard cohorts without retraining or relabeling them as stroke.

The output is a stroke-likeness stress test: how often does the binary model assign a non-CVA neurological or orthopedic gait a high stroke probability? These cohorts are not used to calculate the primary binary AUROC.

In [1]:
import json
import random
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from features import voisard

PROCESSED = PROJECT_ROOT / 'data' / 'processed'
TARGET_FS_HZ = 100.0
WINDOW_SAMPLES = 500
HOP_SAMPLES = 250
NON_CVA = {'CIPN', 'PD', 'RIL', 'ACL', 'HOA', 'KOA'}
ALL_COHORTS = {'healthy': ['HS'], 'neuro': ['CIPN', 'CVA', 'PD', 'RIL'], 'ortho': ['ACL', 'HOA', 'KOA']}
DEVICE = torch.device('cpu')
random.seed(42)
np.random.seed(42)
print('Project root:', PROJECT_ROOT)
print('Device:', DEVICE)

Project root: C:\Users\frank\Documents\MR-ICT Review Paper
Device: cpu


In [2]:
def voisard_bounds(meta):
    uturn_start, uturn_end = meta['uturnBoundaries']
    events = []
    for event_name in ['leftGaitEvents', 'rightGaitEvents']:
        events.extend(meta.get(event_name) or [])
    pre = [event for event in events if event[1] < uturn_start]
    post = [event for event in events if event[0] > uturn_end]
    bounds = []
    if pre:
        bounds.append((min(event[0] for event in pre), max(event[1] for event in pre)))
    if post:
        bounds.append((min(event[0] for event in post), max(event[1] for event in post)))
    return sorted(bounds)

def load_voisard_signal(row):
    trial_dir = Path(row.trial_dir)
    meta = json.loads(Path(row.meta_path).read_text(encoding='utf-8'))
    frames = []
    for sensor in ['LB', 'LF', 'RF']:
        frame = pd.read_csv(trial_dir / f'{row.trial_id}_raw_data_{sensor}.txt', sep='\t')
        values = frame[['Acc_X', 'Acc_Y', 'Acc_Z', 'Gyr_X', 'Gyr_Y', 'Gyr_Z']].astype(float).to_numpy()
        values[:, :3] /= 9.80665
        values[:, 3:] *= 180.0 / np.pi
        frames.append(values)
    n_samples = min(len(frame) for frame in frames)
    return np.concatenate([frame[:n_samples] for frame in frames], axis=1), meta

trials = voisard.list_trials(cohorts=ALL_COHORTS)
trials = trials[trials['cohort'].isin(NON_CVA)].reset_index(drop=True)
window_arrays = []
window_rows = []
trial_rows = []
for row in trials.itertuples(index=False):
    signal, meta = load_voisard_signal(row)
    bounds = voisard_bounds(meta)
    starts = [start for begin, end in bounds for start in range(int(begin), int(end) - WINDOW_SAMPLES + 1, HOP_SAMPLES) if start + WINDOW_SAMPLES <= len(signal)]
    trial_rows.append({'cohort': row.cohort, 'subject': row.subject, 'trial_id': row.trial_id, 'windows': len(starts), 'n_samples': len(signal)})
    for start in starts:
        window_arrays.append(np.column_stack([
            np.linalg.norm(signal[start:start + WINDOW_SAMPLES, 0:3], axis=1),
            np.linalg.norm(signal[start:start + WINDOW_SAMPLES, 6:9], axis=1),
            np.linalg.norm(signal[start:start + WINDOW_SAMPLES, 12:15], axis=1),
        ]).astype('float32'))
        window_rows.append({'cohort': row.cohort, 'subject': row.subject, 'participant_key': f'voisard_2025:{row.subject}', 'trial_id': row.trial_id, 'start_sample': int(start)})

X = np.stack(window_arrays).astype('float32')
window_metadata = pd.DataFrame(window_rows)
trial_audit = pd.DataFrame(trial_rows)
print('Trials:', len(trials), 'Participants:', trials['subject'].nunique())
print('Windows:', X.shape)
print(trial_audit.groupby('cohort').agg(trials=('trial_id', 'nunique'), windows=('windows', 'sum'), participants=('subject', 'nunique')))

Trials: 868 Participants: 138
Windows: (5340, 500, 3)
        trials  windows  participants
cohort                               
ACL         60      166            11
CIPN        98      548            19
HOA         74      363            15
KOA         78      348            18
PD         160      876            24
RIL        398     3039            51


In [3]:
class GaitCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv1d(3, 32, kernel_size=9, padding=4), nn.BatchNorm1d(32), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(32, 64, kernel_size=7, padding=3), nn.BatchNorm1d(64), nn.GELU(), nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=5, padding=2), nn.BatchNorm1d(128), nn.GELU(), nn.AdaptiveAvgPool1d(1),
        )
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.30), nn.Linear(128, 1))

    def forward(self, x):
        return self.classifier(self.features(x)).squeeze(1)

checkpoint_paths = sorted(PROCESSED.glob('cnn_magnitude_fold_*_best.pt'))
assert len(checkpoint_paths) == 5, checkpoint_paths
fold_window_predictions = []
for checkpoint_path in checkpoint_paths:
    checkpoint = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
    model = GaitCNN().to(DEVICE)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    mean = np.asarray(checkpoint['mean'], dtype='float32').reshape(1, 1, 3)
    std = np.asarray(checkpoint['std'], dtype='float32').reshape(1, 1, 3)
    with torch.no_grad():
        for start in range(0, len(X), 512):
            batch = ((X[start:start + 512] - mean) / std).transpose(0, 2, 1).copy()
            probabilities = torch.sigmoid(model(torch.from_numpy(batch))).numpy()
            fold_window_predictions.extend(
                {'fold': int(checkpoint['fold']), 'window_index': i, 'probability': float(p)}
                for i, p in zip(range(start, start + len(probabilities)), probabilities)
            )

predictions = pd.DataFrame(fold_window_predictions).merge(window_metadata.reset_index(names='window_index'), on='window_index', how='left')
participant_fold = predictions.groupby(['fold', 'participant_key', 'cohort', 'subject'], as_index=False).agg(probability=('probability', 'mean'), windows=('window_index', 'nunique'))
participant_predictions = participant_fold.groupby(['participant_key', 'cohort', 'subject'], as_index=False).agg(probability_mean=('probability', 'mean'), probability_sd=('probability', 'std'), windows=('windows', 'mean'))
participant_predictions['predicted_stroke_at_0_5'] = participant_predictions['probability_mean'] >= 0.5
summary = participant_predictions.groupby('cohort', as_index=False).agg(
    participants=('subject', 'nunique'),
    mean_stroke_probability=('probability_mean', 'mean'),
    median_stroke_probability=('probability_mean', 'median'),
    q90_stroke_probability=('probability_mean', lambda x: x.quantile(0.90)),
    fraction_predicted_stroke=('predicted_stroke_at_0_5', 'mean'),
)
print(summary.round(3).to_string(index=False))

cohort  participants  mean_stroke_probability  median_stroke_probability  q90_stroke_probability  fraction_predicted_stroke
   ACL            11                    0.220                      0.214                   0.383                      0.091
  CIPN            19                    0.441                      0.448                   0.717                      0.421
   HOA            15                    0.226                      0.189                   0.404                      0.067
   KOA            18                    0.227                      0.185                   0.456                      0.056
    PD            24                    0.449                      0.505                   0.671                      0.500
   RIL            51                    0.502                      0.544                   0.806                      0.569


In [4]:
predictions.to_csv(PROCESSED / 'non_cva_voisard_window_predictions.csv', index=False)
participant_predictions.to_csv(PROCESSED / 'non_cva_voisard_participant_predictions.csv', index=False)
summary.to_csv(PROCESSED / 'non_cva_voisard_specificity_summary.csv', index=False)
trial_audit.to_csv(PROCESSED / 'non_cva_voisard_trial_audit.csv', index=False)
print('Saved secondary validation outputs in data/processed/')

Saved secondary validation outputs in data/processed/


## Interpretation rule

A high probability on a non-CVA neurological cohort is not a model failure by itself: gait abnormalities can be shared across diseases. It is evidence that the binary classifier may be detecting general gait impairment rather than stroke-specific gait. The primary model should therefore not be claimed as stroke-specific unless it is also compared against these secondary cohorts and, later, an independent clinical cohort.

The 0.5 threshold is inherited from the pilot and is used only for a stress-test summary. It is not a clinical decision threshold.